# LIME internals — verifying the claims behind the lecture

Technical companion to `lime_walkthrough.ipynb`. No teaching figures
here: only commented code, prints, and small tables. It answers four
questions that the lecture notebook asserts but does not prove.

1. Does our description of the algorithm match the installed source code
   of the `lime` package — perturbation, kernel, and feature selection?
2. How was the patient in the lecture chosen, and how many alternatives
   were there?
3. Does a high $R^2$ actually mean a better explanation?
4. How stable is an explanation across repeated runs?

Runtime: the sweeps in §3 and §4 each call `explain_instance` once per
test patient (143 × 5,000 samples), so a full run takes a few minutes.

## Setup

In [1]:
%pip install -q lime scikit-learn numpy scipy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
from scipy.stats import spearmanr

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import accuracy_score, roc_auc_score

from lime.lime_tabular import LimeTabularExplainer

RANDOM_STATE = 42

## 1 · Model and dataset (identical to the lecture notebook)

In [3]:
data = load_breast_cancer()
X_all, y_all = data.data, data.target
feature_names = list(data.feature_names)
class_names = list(data.target_names)  # ['malignant', 'benign']
feat_idx = {f: i for i, f in enumerate(feature_names)}

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.25, random_state=RANDOM_STATE, stratify=y_all
)
model = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=RANDOM_STATE)
model.fit(X_train, y_train)

proba_test = model.predict_proba(X_test)[:, 1]
print(f"test accuracy = {accuracy_score(y_test, model.predict(X_test)):.1%}   "
      f"ROC-AUC = {roc_auc_score(y_test, proba_test):.3f}")

train_mean = X_train.mean(axis=0)
train_std = X_train.std(axis=0)
kernel_width = 0.75 * np.sqrt(len(feature_names))  # lime_tabular.py:243 — the default, untouched

test accuracy = 95.1%   ROC-AUC = 0.993


## 2 · Geometry helpers

For any axis pair `(ix, iy)`: the real boundary of the 30-feature model
sliced along that plane (the other 28 features fixed at the patient's own
values), and the distance from the patient to the local fit's $P=0.5$
line.

The distance to the line has a closed form worth knowing, because it
explains a trap we fell into while building this material. In
standardized space the line is $w_x z_x + w_y z_y + c = 0.5$, so

$$\text{distance} = \frac{\lvert g(x) - 0.5 \rvert}{\lVert (w_x, w_y) \rVert}$$

The numerator is how far the surrogate's own prediction at the patient is
from $0.5$. **For a patient the model is confident about, that numerator
is large and the line is therefore forced far away from her** — no
choice of axes can fix it. That is why the lecture requires a
boundary-adjacent patient (§3).

Crucial implementation detail: the package fits the local model on
STANDARDIZED data — `lime_tabular.py:452-454` passes `scaled_data`
(line 348: `(data - mean) / scale`) into `explain_instance_with_data`,
not the raw units. So `local_exp` / `intercept` live in standardized
space and any geometry built from them must convert in and out.

In [4]:
def axis_grid(i, n=60):
    """The real value range of feature i, with an 8% margin on each side."""
    lo, hi = X_all[:, i].min(), X_all[:, i].max()
    pad = 0.08 * (hi - lo)
    return np.linspace(lo - pad, hi + pad, n)


def boundary_slice(base_row, ix, iy, grid_x, grid_y):
    GX, GY = np.meshgrid(grid_x, grid_y)
    points = np.tile(base_row, (GX.size, 1))
    points[:, ix] = GX.ravel()
    points[:, iy] = GY.ravel()
    return GX, GY, model.predict_proba(points)[:, 1].reshape(GX.shape)


def dist_to_boundary(base_row, ix, iy):
    """Smallest distance (in σ, measured in the 2-D plane) from the patient
    to the P=0.5 contour of the real model. NaN if the slice never crosses."""
    gx = np.linspace(base_row[ix] - 3 * train_std[ix], base_row[ix] + 3 * train_std[ix], 120)
    gy = np.linspace(base_row[iy] - 3 * train_std[iy], base_row[iy] + 3 * train_std[iy], 120)
    GX, GY, P = boundary_slice(base_row, ix, iy, gx, gy)
    side = P >= 0.5
    if side.all() or (~side).all():
        return np.nan
    edge = np.zeros_like(side)
    edge[:, :-1] |= side[:, :-1] != side[:, 1:]
    edge[:-1, :] |= side[:-1, :] != side[1:, :]
    return float(np.min(np.hypot((GX[edge] - base_row[ix]) / train_std[ix],
                                 (GY[edge] - base_row[iy]) / train_std[iy])))


def dist_to_line(exp, ix, iy):
    """Distance (in σ) from the patient to the local fit's P=0.5 line."""
    cm = dict(exp.local_exp[1])
    if ix not in cm or iy not in cm:
        return np.nan
    grad = np.hypot(cm[ix], cm[iy])
    return float(abs(exp.local_pred[0] - 0.5) / grad) if grad > 1e-12 else np.nan

## 3 · Choosing the patient for the lecture

A figure is only didactic if the geometry in it means something. Three
requirements, each traceable to a way the picture can fail:

| requirement | what goes wrong without it |
|---|---|
| the model classifies her correctly | the lesson gets tangled up with a model error |
| $\lvert P - 0.5\rvert < 0.15$ (near the boundary) | by the formula in §2, the fit's line is pushed far from the patient; the plot shows two unrelated lines |
| top-2 features correlate at $\lvert r\rvert < 0.6$ | the two axes are near-duplicates, real patients collapse onto a 1-D ribbon, and there is no 2-D structure to see |

The sweep below applies them to all 143 test patients. Note how few
survive — and note, in the full ranking, that the highest-$R^2$ patients
are exactly the ones whose geometry is useless.

One reading note: this sweep reuses a single explainer across all
patients, so its internal random state advances and the per-patient $R^2$
printed here differs slightly from the fresh single-call value in §5
(0.27 vs 0.36 for the lecture patient, in the committed run). That gap is
not a bug to be tidied away — it is the very instability quantified in §9,
showing up as a side effect.

In [5]:
explainer_sweep = LimeTabularExplainer(
    X_train, feature_names=feature_names, class_names=class_names,
    discretize_continuous=False, random_state=RANDOM_STATE,
)

rows = []
for idx in range(len(X_test)):
    cand = X_test[idx]
    p = proba_test[idx]
    e = explainer_sweep.explain_instance(cand, model.predict_proba,
                                         num_features=8, num_samples=5000, labels=(1,))
    (cix, _), (ciy, _) = sorted(e.local_exp[1], key=lambda t: abs(t[1]), reverse=True)[:2]
    rows.append({
        "idx": idx,
        "P": p,
        "correct": bool((p >= 0.5) == y_test[idx]),
        "r2": e.score,
        "axes": (feature_names[cix], feature_names[ciy]),
        "corr": float(np.corrcoef(X_train[:, cix], X_train[:, ciy])[0, 1]),
        "d_line": dist_to_line(e, cix, ciy),
        "d_bnd": dist_to_boundary(cand, cix, ciy),
    })

survivors = [r for r in rows
             if r["correct"] and abs(r["P"] - 0.5) < 0.15 and abs(r["corr"]) < 0.6]

print(f"test patients:                                     {len(rows)}")
print(f"  correctly classified:                            {sum(r['correct'] for r in rows)}")
print(f"  ... and near the boundary (|P-0.5| < 0.15):      "
      f"{sum(r['correct'] and abs(r['P']-0.5) < 0.15 for r in rows)}")
print(f"  ... and with decorrelated axes (|r| < 0.6):      {len(survivors)}")
print("\nsurvivors:")
for r in survivors:
    print(f"  #{r['idx']:3d}  P={r['P']:.3f}  R²={r['r2']:.2f}  "
          f"boundary {r['d_bnd']:.2f}σ away  line {r['d_line']:.2f}σ away  "
          f"corr={r['corr']:+.2f}  {r['axes']}")

print("\nfor contrast, the five HIGHEST-R² patients (any confidence):")
for r in sorted(rows, key=lambda r: -r["r2"])[:5]:
    print(f"  #{r['idx']:3d}  P={r['P']:.3f}  R²={r['r2']:.2f}  "
          f"line {r['d_line']:.2f}σ away  corr={r['corr']:+.2f}")
print("→ high R² comes with confident predictions, distant lines, and "
      "near-duplicate axes: a well-fitted explanation of nothing much.")

test patients:                                     143
  correctly classified:                            136
  ... and near the boundary (|P-0.5| < 0.15):      4
  ... and with decorrelated axes (|r| < 0.6):      1

survivors:
  # 67  P=0.581  R²=0.27  boundary 0.08σ away  line 0.00σ away  corr=+0.35  (np.str_('worst perimeter'), np.str_('worst texture'))

for contrast, the five HIGHEST-R² patients (any confidence):
  # 66  P=0.037  R²=0.86  line 7.17σ away  corr=+0.98
  # 59  P=0.002  R²=0.69  line 6.66σ away  corr=+0.98
  # 84  P=0.009  R²=0.67  line 4.50σ away  corr=+0.98
  #117  P=1.000  R²=0.65  line 1.88σ away  corr=+0.98
  # 34  P=0.991  R²=0.65  line 2.35σ away  corr=+0.98
→ high R² comes with confident predictions, distant lines, and near-duplicate axes: a well-fitted explanation of nothing much.


The patient used in the lecture is the survivor of that filter. Everything
from here on uses her.

In [6]:
instance_idx = survivors[0]["idx"] if survivors else 67
row = X_test[instance_idx]
row_proba = proba_test[instance_idx]
row_scaled = (row - train_mean) / train_std
ix, iy = feat_idx["worst perimeter"], feat_idx["worst texture"]
print(f"lecture patient: #{instance_idx}  P(benign)={row_proba:.3f}  "
      f"true={class_names[y_test[instance_idx]]}")
if instance_idx != 67:
    print("WARNING: this run selected a different patient than the committed lecture "
          "material (#67), most likely because of differing library versions.")

lecture patient: #67  P(benign)=0.581  true=benign


## 4 · Does a high $R^2$ mean a better explanation?

The lecture claims it does not — that $R^2$ largely reports how saturated
the model is at that point, not how insightful the explanation is. That
is a testable claim, and this is the test: correlate each patient's
fidelity against how confident the model was about her.

The mechanism, if the claim holds: far from the boundary $f$ barely
moves, and a straight line reproduces "almost constant" very well.

In [7]:
confidence = np.array([abs(r["P"] - 0.5) for r in rows])
fidelity = np.array([r["r2"] for r in rows])
rho, pval = spearmanr(confidence, fidelity)
print(f"Spearman correlation between |P-0.5| and R²: ρ = {rho:+.3f}  (p = {pval:.1e}, n = {len(rows)})")
print()
for lo, hi, label in [(0.0, 0.15, "borderline  |P-0.5| < 0.15"),
                      (0.15, 0.30, "intermediate"),
                      (0.30, 0.51, "confident   |P-0.5| > 0.30")]:
    sel = (confidence >= lo) & (confidence < hi)
    if sel.sum():
        print(f"  {label:28s} n={sel.sum():3d}   mean R² = {fidelity[sel].mean():.3f}   "
              f"(range {fidelity[sel].min():.2f}–{fidelity[sel].max():.2f})")
print("\n→ fidelity rises with confidence. Reported on its own, R² rewards "
      "explanations of saturated regions and penalizes the borderline cases "
      "where an explanation is actually worth having.")

Spearman correlation between |P-0.5| and R²: ρ = +0.572  (p = 8.2e-14, n = 143)

  borderline  |P-0.5| < 0.15   n=  6   mean R² = 0.448   (range 0.27–0.55)
  intermediate                 n= 10   mean R² = 0.553   (range 0.34–0.64)
  confident   |P-0.5| > 0.30   n=127   mean R² = 0.600   (range 0.16–0.86)

→ fidelity rises with confidence. Reported on its own, R² rewards explanations of saturated regions and penalizes the borderline cases where an explanation is actually worth having.


## 5 · The official explanation, spied on

A fresh explainer, `explain_instance` called once, with `predict_proba`
swapped for a spy that records the synthetic neighborhood generated
internally. Everything in §6-8 reuses exactly that sample, so the
comparisons are not polluted by resampling noise.

In [8]:
captured = {}


def spy_predict_proba(X_in):
    captured["X"] = np.array(X_in, dtype=float).copy()
    return model.predict_proba(X_in)


explainer = LimeTabularExplainer(
    X_train, feature_names=feature_names, class_names=class_names,
    discretize_continuous=False, random_state=RANDOM_STATE,
)
exp_official = explainer.explain_instance(
    row, spy_predict_proba, num_features=8, num_samples=5000, labels=(1,)
)
Z_captured = captured["X"]
proba_captured = model.predict_proba(Z_captured)[:, 1]
print(f"captured neighborhood: {Z_captured.shape[0]} samples × {Z_captured.shape[1]} features")
print(f"official R² = {exp_official.score:.3f}")

captured neighborhood: 5000 samples × 30 features
official R² = 0.357


## 6 · Perturbation vs `__data_inverse`

`lime_tabular.py:511-517`: for each continuous feature, draw
$\mathcal{N}(0,1)$ and undo the standardization —
`data = data*scale + mean`, with `scale`/`mean` the TRAINING std/mean
(`StandardScaler(with_mean=False)`, lines 257-258). This holds because
`sample_around_instance=False` is the default (line 138); otherwise it
would be `+ instance_sample` (line 515), centred on the patient rather
than on the training mean.

So the neighborhood is centred on the *dataset*, not on the patient — a
detail with real consequences for how large "local" is.

In [9]:
mean_err = np.abs((Z_captured.mean(axis=0) - train_mean) / train_std).max()
std_err = np.abs((Z_captured.std(axis=0) - train_std) / train_std).max()
print(f"largest deviation of the captured mean from train_mean: {mean_err:.1%} of σ")
print(f"largest deviation of the captured std  from train_std:  {std_err:.1%} of σ")
print("→ a few percent, the size expected from sampling noise with 5,000 draws "
      "in 30 dimensions. The formula matches.")

largest deviation of the captured mean from train_mean: 3.0% of σ
largest deviation of the captured std  from train_std:  3.6% of σ
→ a few percent, the size expected from sampling noise with 5,000 draws in 30 dimensions. The formula matches.


## 7 · The proximity kernel

`lime_tabular.py:243`: `kernel_width = sqrt(n_features) * 0.75`.
`lime_tabular.py:248`: $\pi = \sqrt{\exp(-d^2/\nu^2)}$ — note the outer
square root, which makes it $\exp(-d^2/2\nu^2)$, a Gaussian of width
$\nu\sqrt{2}$. Distances are computed on the standardized features.

In [10]:
Z_scaled = (Z_captured - train_mean) / train_std
dist = np.linalg.norm(Z_scaled - row_scaled, axis=1)
weight = np.sqrt(np.exp(-(dist ** 2) / (kernel_width ** 2)))

print(f"kernel width ν = {kernel_width:.2f} (in 30-D standardized units)")
print(f"median neighbor distance: {np.median(dist):.2f}σ   →   median weight: "
      f"{np.median(weight):.3f}")
print(f"fraction of neighbors with weight > 0.5: {(weight > 0.5).mean():.0%}")
print("→ the typical neighbor sits several σ away in 30-D and still carries "
      "roughly a third of the maximum weight. Few neighbors are 'close' in any "
      "strict sense, yet the fit is dominated by the many mid-weight ones, so "
      "'local' here really means a large slice of the data space — which is why "
      "the R² of §4 is a statement about that whole region, not about the "
      "immediate vicinity of the patient.")

kernel width ν = 4.11 (in 30-D standardized units)
median neighbor distance: 6.18σ   →   median weight: 0.323
fraction of neighbors with weight > 0.5: 4%
→ the typical neighbor sits several σ away in 30-D and still carries roughly a third of the maximum weight. Few neighbors are 'close' in any strict sense, yet the fit is dominated by the many mid-weight ones, so 'local' here really means a large slice of the data space — which is why the R² of §4 is a statement about that whole region, not about the immediate vicinity of the patient.


## 8 · The `highest_weights` feature selection

`lime_base.py:109` (dense case): `weighted_data = coef * data[0]` — an
auxiliary Ridge (`alpha=0.01`) on ALL features, weighted by the same
kernel; each feature scores $\lvert \text{coef} \times \text{value}\rvert$;
the top `num_features` are kept.

The trap: that `data[0]` is `scaled_data[0]`, the **standardized** patient,
not her raw values. Reproducing this in raw units silently yields a
different top-8 — we made exactly that mistake while building this
material, and it corrupted the plotted geometry before it was caught.

In [11]:
aux = Ridge(alpha=0.01).fit(Z_scaled, proba_captured, sample_weight=weight)
score_std = np.abs(aux.coef_ * row_scaled)          # correct: standardized
score_raw = np.abs(aux.coef_ * row)                 # the bug: raw units
top8_std = set(np.argsort(-score_std)[:8])
top8_raw = set(np.argsort(-score_raw)[:8])
top8_official = set(f for f, _ in exp_official.local_exp[1])

print(f"reproduced in standardized space -> matches official: {top8_std == top8_official} "
      f"({len(top8_std & top8_official)}/8 in common)")
print(f"reproduced in raw units          -> matches official: {top8_raw == top8_official} "
      f"({len(top8_raw & top8_official)}/8 in common)")
print("\nofficial weights (negative = pushes toward malignant):")
for f, w in sorted(exp_official.local_exp[1], key=lambda t: abs(t[1]), reverse=True):
    print(f"  {feature_names[f]:<26} {w:+.4f}")

reproduced in standardized space -> matches official: True (8/8 in common)
reproduced in raw units          -> matches official: False (5/8 in common)

official weights (negative = pushes toward malignant):
  worst perimeter            -0.0709
  worst concave points       -0.0457
  worst texture              -0.0213
  area error                 -0.0184
  mean texture               -0.0153
  mean perimeter             -0.0140
  mean radius                -0.0134
  radius error               -0.0089


`lime_base.py:194-196` confirms that the reported `score` is itself
weighted (`easy_model.score(..., sample_weight=weights)`), and
`lime_base.py:204-206` that `local_exp` arrives sorted by decreasing
$\lvert w\rvert$ — which is why taking its first two entries in §3 gives
the top-2 axes directly.

## 9 · Stability across runs

Molnar flags instability as one of LIME's serious limitations. Each
`explain_instance` draws a fresh sample, so the same patient can yield
different explanations. Below: four independent redraws of the lecture
patient, comparing both the selected feature set and the fidelity.

In [12]:
base_top8 = [f for f, _ in sorted(exp_official.local_exp[1], key=lambda t: abs(t[1]), reverse=True)]
print(f"reference (seed {RANDOM_STATE}):  R² = {exp_official.score:.3f}   "
      f"top feature = {feature_names[base_top8[0]]}")
for s in range(1, 5):
    e = LimeTabularExplainer(
        X_train, feature_names=feature_names, class_names=class_names,
        discretize_continuous=False, random_state=RANDOM_STATE + s,
    ).explain_instance(row, model.predict_proba, num_features=8, num_samples=5000, labels=(1,))
    t8 = set(f for f, _ in e.local_exp[1])
    top1 = max(e.local_exp[1], key=lambda t: abs(t[1]))[0]
    print(f"  redraw (seed {RANDOM_STATE + s}): R² = {e.score:.3f}   "
          f"{len(t8 & top8_official)}/8 features shared   "
          f"top feature = {feature_names[top1]}")
print("\n→ the feature set wobbles at the margins, mostly among near-duplicate "
      "size measurements, while the leading feature and the direction of the "
      "effect stay put. Report the direction; do not over-read the exact ranking.")

reference (seed 42):  R² = 0.357   top feature = worst perimeter
  redraw (seed 43): R² = 0.294   7/8 features shared   top feature = worst perimeter
  redraw (seed 44): R² = 0.278   7/8 features shared   top feature = worst perimeter


  redraw (seed 45): R² = 0.282   7/8 features shared   top feature = worst perimeter


  redraw (seed 46): R² = 0.305   7/8 features shared   top feature = worst perimeter

→ the feature set wobbles at the margins, mostly among near-duplicate size measurements, while the leading feature and the direction of the effect stay put. Report the direction; do not over-read the exact ranking.


## Summary

- Perturbation, kernel, and feature selection reproduce the installed
  `lime` source exactly — provided the selection is done in standardized
  space (§8).
- The lecture patient is the unique test case satisfying all three
  didactic requirements; high-$R^2$ patients systematically fail them
  (§3).
- $R^2$ correlates with prediction confidence, so it must not be read as
  an explanation-quality score on its own (§4).
- The default neighborhood is wide (§7) and explanations wobble across
  redraws (§9) — the two limitations Molnar singles out, both visible
  here in numbers.

In [13]:
print("Lecture version, with the figures: lime_walkthrough.ipynb")

Lecture version, with the figures: lime_walkthrough.ipynb
